# 09 · Enterprise Delta Lake: ACID, Time Travel & Disaster Recovery
**Table under audit:** `vstone_catalog.silver.listings_silver_merged`

| Section | Concept demonstrated |
|---|---|
| **1 · Setup** | Config, helpers, pre-run version snapshot |
| **2 · Audit Trail** | `DESCRIBE HISTORY` — who, when, what, metrics |
| **3 · Time Travel** | Version 0 vs current — schema evolution proof |
| **4 · Disaster Recovery** | Simulate corruption → Time Travel read → compensating `MERGE` |
| **5 · Business Rule Checks** | Date parsing, RUB→USD, price segmentation |
| **6 · Integrity Summary** | Row-count audit across all Silver tables |



## 1 · Setup & Configuration

In [0]:
# ── 1. CONFIGURATION ─────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from delta.tables import DeltaTable

CATALOG    = "vstone_catalog"
BRONZE     = "bronze"
SILVER     = "silver"
TABLE_NAME = "listings_silver_merged"
FQN        = f"{CATALOG}.{SILVER}.{TABLE_NAME}"

USD_RATE = 82.5   # Feb 2023 historical RUB/USD rate (mirrors 08_silver_transformation)

# ── Console helpers ───────────────────────────────────────────────────────────
def _header(title: str):
    bar = "=" * 64
    print(f"\n{bar}\n  {title}\n{bar}")

def _ok(msg):   print(f"  ✓  {msg}")
def _warn(msg): print(f"  ⚠  {msg}")
def _fail(msg): print(f"  ✗  {msg}")

# ── Pre-flight: find the last clean DLT version (STREAMING UPDATE) ────────────
# _pre_version must be a genuine DLT data commit, not a manual UPDATE/MERGE
# from a previous run of this notebook. We scan DESCRIBE HISTORY and take the
# most recent version whose operation = 'STREAMING UPDATE' — that is guaranteed
# to contain original pipeline-written values, never corrupted by ad-hoc DML.
_history_rows = (
    spark.sql(f"DESCRIBE HISTORY {FQN}")
    .select("version", "operation")
    .orderBy("version", ascending=False)
    .collect()
)

_pre_version = None
for row in _history_rows:
    if row["operation"] == "STREAMING UPDATE":
        _pre_version = row["version"]
        break

if _pre_version is None:
    raise RuntimeError(
        "No STREAMING UPDATE found in history — "
        "cannot determine a clean DLT baseline version."
    )

_header("CONFIG LOADED")
print(f"  Table              : {FQN}")
print(f"  USD rate           : 1 USD = {USD_RATE} RUB")
print(f"  Last DLT version   : {_pre_version}  ← clean STREAMING UPDATE baseline")


## 2 · DML Audit Trail
Delta Lake records every atomic commit in `_delta_log/`. `DESCRIBE HISTORY` surfaces the full lineage — actor, timestamp, operation, predicates, and file-level metrics. This is the primary compliance artefact in regulated environments.

In [0]:
# ── 2. AUDIT TRAIL — DESCRIBE HISTORY ────────────────────────────────────────
# Delta Lake records every atomic commit in its transaction log (_delta_log/).
# Each row = one operation: who triggered it, when, with what parameters,
# and how many files / rows were affected.
# In a regulated environment this log is the primary compliance artefact.

_header("SECTION 2 · DML AUDIT TRAIL  (DESCRIBE HISTORY)")

audit_df = (
    spark.sql(f"DESCRIBE HISTORY {FQN}")
    .select(
        "version",
        "timestamp",
        "userName",             # identity of the actor (IAM principal)
        "operation",            # WRITE | UPDATE | MERGE | RESTORE | …
        "operationParameters",  # predicates / output-mode details
        "operationMetrics",     # rows written, files added / removed
    )
    .orderBy("version", ascending=False)
)

total_versions = audit_df.count()
print(f"  Total committed versions: {total_versions}")
print()
display(audit_df)


## 3 · Time Travel — Earliest Data Version vs Current
Delta Lake retains every historical snapshot without additional storage overhead. DLT writes **Version 0 as a schema-init commit** (empty, no columns) before data arrives — so we scan forward from version 0 to find the first version with actual rows. We then diff its schema against the current state to demonstrate column evolution, and read both snapshots side-by-side as proof of point-in-time reproducibility.

In [0]:
# ── 3. TIME TRAVEL — EARLIEST DATA VERSION vs CURRENT ────────────────────────
# Delta Lake retains every historical snapshot in the transaction log (_delta_log/).
# DLT pipelines write Version 0 as a schema-only initialisation commit (no rows,
# no columns) — the first version with real data is Version 1.
# We scan the history to find the earliest version that actually has columns,
# then compare it with the current state to demonstrate schema evolution and
# point-in-time reproducibility.

_header("SECTION 3 · TIME TRAVEL  (EARLIEST DATA VERSION vs CURRENT)")

# ── 3a. Discover the earliest readable version (has columns + rows) ───────────
# DLT Version 0 = schema-init commit with no columns.  We walk forward from 0
# until we find the first version that Spark can actually read as a DataFrame.
all_versions = [
    row["version"]
    for row in spark.sql(f"DESCRIBE HISTORY {FQN}")
    .select("version")
    .orderBy("version")          # ascending → oldest first
    .collect()
]

df_earliest = None
earliest_ver = None

for ver in all_versions:
    try:
        candidate = spark.read.format("delta").option("versionAsOf", ver).table(FQN)
        if len(candidate.columns) > 0:          
            df_earliest  = candidate
            earliest_ver = ver
            break
    except Exception:
        continue                                 

if df_earliest is not None:
    early_count = df_earliest.count()
    early_cols  = df_earliest.columns
    _ok(f"Earliest data version: {earliest_ver} — {early_count:,} rows | {len(early_cols)} columns")
else:
    _warn("No readable historical version found — log may have been vacuumed.")

# ── 3b. Current snapshot ──────────────────────────────────────────────────────
df_cur    = spark.table(FQN)
cur_count = df_cur.count()
cur_cols  = df_cur.columns
_ok(f"Current version loaded — {cur_count:,} rows | {len(cur_cols)} columns")

# ── 3c. Schema evolution diff ─────────────────────────────────────────────────
if df_earliest is not None:
    added   = [c for c in cur_cols  if c not in early_cols]
    removed = [c for c in early_cols if c not in cur_cols]
    print()
    print(f"  Columns V{earliest_ver}       : {list(early_cols[:8])} …")
    print(f"  Columns current   : {list(cur_cols[:8])}  …")
    print(f"  Added since V{earliest_ver}   : {added   or 'none'}")
    print(f"  Removed since V{earliest_ver} : {removed or 'none'}")

    # ── 3d. Side-by-side sample ───────────────────────────────────────────────
    print()
    print(f"  [PAST V{earliest_ver}]  Earliest data snapshot (listing_id, listing_date, price_rub):")
    df_earliest.select("listing_id", "listing_date", "price_rub").show(5, truncate=False)

print("  [CURRENT]  Latest version sample:")
df_cur.select("listing_id", "listing_date", "price_rub", "brand", "price_category")     .show(5, truncate=False)

print()
_ok(f"Time travel verified — any version from {earliest_ver} to {_pre_version} is reproducible via versionAsOf.")


## 4 · Disaster Recovery — Simulate Corruption → Compensating MERGE
An accidental `UPDATE` corrupts brand values for all 2020 listings. `RESTORE TABLE` is **not supported on DLT Streaming Tables**, so the correct production pattern is a **compensating MERGE**: read the original values from the clean baseline snapshot via Time Travel (`versionAsOf`), then MERGE them back as a new forward commit. This is actually more auditable than RESTORE — both the corruption and the fix appear as separate versions in `DESCRIBE HISTORY`.

In [0]:
# ── 4. DISASTER RECOVERY — SIMULATE CORRUPTION → COMPENSATING MERGE ───────────
# RESTORE TABLE is not supported on DLT Streaming Tables.
# The production-correct recovery pattern for Streaming Tables is:
#   1. Read original values from the pre-corruption snapshot via TIME TRAVEL.
#   2. Apply a compensating MERGE that writes correct values back as a NEW commit.
# This is actually BETTER than RESTORE for auditability — every change is
# recorded as a forward commit, preserving the full change history.

_header("SECTION 4 · DISASTER RECOVERY  (UPDATE corruption → TIME TRAVEL read → MERGE recovery)")

# ── 4a. Baseline ──────────────────────────────────────────────────────────────
baseline_count = spark.table(FQN).count()
_ok(f"Baseline row count: {baseline_count:,}")

# ── 4b. Simulate accidental data corruption ───────────────────────────────────
print()
print("  Simulating accidental UPDATE: brand = 'CORRUPTED_DATA' for manufacture_year = 2020 …")
spark.sql(
    f"UPDATE {FQN} SET brand = 'CORRUPTED_DATA' WHERE manufacture_year = 2020"
)

corrupt_count = spark.sql(
    f"SELECT COUNT(*) FROM {FQN} WHERE brand = 'CORRUPTED_DATA'"
).collect()[0][0]
_fail(f"ALERT: {corrupt_count:,} rows corrupted in production!")

# ── 4c. Read original values from the last DLT commit via Time Travel ─────────
# _pre_version = last STREAMING UPDATE (found in Cell 1).
# Guaranteed to pre-date any manual corruption even across multiple notebook runs,
# because _pre_version always resolves to the most recent genuine DLT data commit.
clean_version = _pre_version
print(f"\n  Reading original brand values from Version {clean_version} (last STREAMING UPDATE) …")

df_original = (
    spark.read.format("delta")
    .option("versionAsOf", clean_version)
    .table(FQN)
    .filter("manufacture_year = 2020")
    .select("listing_id", "brand")   # PK + column to restore
)
_ok(f"Loaded {df_original.count():,} original brand values from Version {clean_version}")
df_original.createOrReplaceTempView("_recovery_brands")

# ── 4d. Compensating MERGE — overwrite corrupted values with originals ─────────
print("  Applying compensating MERGE to restore original brand values …")
merge_sql = (
    f"MERGE INTO {FQN} AS target "
    "USING _recovery_brands AS src "
    "ON target.listing_id = src.listing_id "
    "WHEN MATCHED AND target.brand = 'CORRUPTED_DATA' "
    "THEN UPDATE SET target.brand = src.brand"
)
spark.sql(merge_sql)

# ── 4e. Validation ────────────────────────────────────────────────────────────
post_corrupt = spark.sql(
    f"SELECT COUNT(*) FROM {FQN} WHERE brand = 'CORRUPTED_DATA'"
).collect()[0][0]
post_count   = spark.table(FQN).count()

print()
if post_corrupt == 0 and post_count == baseline_count:
    _ok(f"RECOVERY SUCCESS — corrupted rows: 0 | row count: {post_count:,} (matches baseline)")
else:
    _fail(f"Recovery incomplete — corrupted rows: {post_corrupt} | count: {post_count:,}")

# ── 4f. Audit trail — corruption + recovery both visible as separate commits ───
print()
print("  Audit trail — last 4 operations (UPDATE corruption + MERGE recovery both recorded):")
spark.sql(f"DESCRIBE HISTORY {FQN}") \
    .select("version", "timestamp", "operation", "operationParameters") \
    .orderBy("version", ascending=False) \
    .limit(4) \
    .show(truncate=80)
print()
print("  KEY INSIGHT: Unlike RESTORE which is blocked on Streaming Tables, the")
print("  compensating MERGE produces a NEW auditable version — every change is")
print("  forward-recorded in the transaction log, meeting compliance requirements.")


## 5 · Business Rule Validation
Three automated checks confirm that Silver enrichment columns were computed correctly by `08_silver_transformation`:
- **5a** `listing_year` / `listing_month` match `listing_date`
- **5b** `price_usd = round(price_rub / 82.5, 2)` within ±0.01 tolerance
- **5c** `price_category` min/max per band stays within defined RUB thresholds

In [0]:
# ── 5. BUSINESS RULE VALIDATION ───────────────────────────────────────────────
# Verifies that Silver enrichment columns match the logic in 08_silver_transformation.
#
#   5a · Date components   — listing_year / listing_month match listing_date
#   5b · Currency          — price_usd = round(price_rub / 82.5, 2)  within ±0.01
#   5c · Price segmentation — category bands stay within defined RUB thresholds
#
# All checks emit a ✓ / ✗ verdict so failures are immediately visible.

_header("SECTION 5 · BUSINESS RULE VALIDATION")

df = spark.table(FQN)
total_rows = df.count()

# ── 5a. Date parsing ──────────────────────────────────────────────────────────
print("  CHECK 5a · listing_year / listing_month derived from listing_date")

date_violations = df.filter(
    F.col("listing_date").isNotNull() & (
        (F.year("listing_date")  != F.col("listing_year"))  |
        (F.month("listing_date") != F.col("listing_month"))
    )
).count()

if date_violations == 0:
    _ok(f"Date parsing correct — 0 violations across {total_rows:,} rows")
else:
    _fail(f"{date_violations:,} rows have mismatched listing_year / listing_month")

df.filter(F.col("listing_date").isNotNull())   .select("listing_date", "listing_year", "listing_month")   .limit(5).show()

# ── 5b. Currency normalization ────────────────────────────────────────────────
print("  CHECK 5b · price_usd = round(price_rub / 82.5, 2)  [tolerance ±0.01]")

usd_violations = df.filter(
    F.col("price_rub").isNotNull() &
    F.col("price_usd").isNotNull() &
    (F.abs(F.col("price_usd") - F.round(F.col("price_rub") / USD_RATE, 2)) > 0.01)
).count()

if usd_violations == 0:
    _ok("Currency conversion correct — 0 violations")
else:
    _fail(f"{usd_violations:,} rows have incorrect price_usd")

df.filter(F.col("price_rub") > 0)   .select(
      "price_rub",
      "price_usd",
      F.round(F.col("price_rub") / USD_RATE, 2).alias("expected_usd"),
  ).limit(5).show()

# ── 5c. Price segmentation ────────────────────────────────────────────────────
print("  CHECK 5c · price_category segmentation bands")

# Thresholds from 08_silver_transformation
PRICE_BANDS = {
    "BUDGET"   : (0,         299_999),
    "MID_RANGE": (300_000,   700_000),
    "PREMIUM"  : (700_001, 1_500_000),
    "LUXURY"   : (1_500_001, float("inf")),
}

seg_df = (
    df.filter(F.col("price_rub").isNotNull())
    .groupBy("price_category")
    .agg(
        F.count("*").alias("row_count"),
        F.min("price_rub").alias("min_price_rub"),
        F.max("price_rub").alias("max_price_rub"),
    )
    .orderBy("min_price_rub")
)
display(seg_df)

seg_violations = 0
for row in seg_df.collect():
    cat = row["price_category"]
    if cat not in PRICE_BANDS:
        continue
    lo, hi = PRICE_BANDS[cat]
    if not (lo <= row["min_price_rub"] and row["max_price_rub"] <= hi):
        _fail(f"{cat}: min={row['min_price_rub']:,.0f} max={row['max_price_rub']:,.0f} outside [{lo:,} – {hi:,}]")
        seg_violations += 1

if seg_violations == 0:
    _ok("Price segmentation correct — all categories within defined thresholds")


## 6 · Integrity Summary
Row-count audit across all 10 Silver + Quarantine tables confirms no rows were lost or invented during this session's DML. The version delta shows exactly how many commits this notebook created.

In [0]:
# ── 6. INTEGRITY SUMMARY ──────────────────────────────────────────────────────
# Row-count audit across all 10 Silver + Quarantine tables.
# Confirms the DML operations in this notebook left no orphaned or missing rows,
# and shows the total version delta created during this session.

_header("SECTION 6 · INTEGRITY SUMMARY")

SILVER_TABLES = [
    ("silver",     "listings_silver_merged"),
    ("quarantine", "listings_main_quarantine"),
    ("silver",     "listings_text_transformation"),
    ("quarantine", "listings_text_quarantine"),
    ("silver",     "listings_photo_transformation"),
    ("quarantine", "listings_photo_quarantine"),
    ("silver",     "car_catalog_transformation"),
    ("quarantine", "car_catalog_quarantine"),
    ("silver",     "geography_transformation"),
    ("quarantine", "geography_quarantine"),
]

total = 0
for layer, tbl in SILVER_TABLES:
    fqn = f"{CATALOG}.{SILVER}.{tbl}"
    cnt = spark.table(fqn).count()
    total += cnt
    tag   = "SILVER    " if layer == "silver" else "QUARANTINE"
    print(f"  {tag}  {tbl:<45}  {cnt:>10,} rows")

print()
_ok(f"All {len(SILVER_TABLES)} tables accessible")
_ok(f"Total rows across Silver layer: {total:,}")

# ── Version delta for this session ────────────────────────────────────────────
final_version = (
    spark.sql(f"DESCRIBE HISTORY {FQN}")
    .select("version")
    .orderBy("version", ascending=False)
    .first()[0]
)
print()
print(f"  listings_silver_merged  pre-run version  : {_pre_version}")
print(f"  listings_silver_merged  final version    : {final_version}")
print(f"  Versions created this session            : {final_version - _pre_version}")
print()
_ok("Notebook complete — Delta Lake ACID, Time Travel, and compensating MERGE recovery demonstrated.")
